# Predios para Conservación y PSA — Análisis de Hectáreas bajo Pago por Servicios Ambientales

**Bloque:** A — Gestión | **Línea temática:** `predios_conservacion` | **Variable principal:** hectáreas bajo PSA (ha)

---

## ¿Qué son los Predios para Conservación y el PSA en Colombia?

La gestión de predios para conservación en Colombia se articula alrededor de dos instrumentos complementarios:

1. **Adquisición predial directa:** El **Artículo 111 de la Ley 99 de 1993** obliga a municipios y departamentos a destinar al menos el **1% de sus ingresos corrientes de libre destinación** para adquirir o mantener las Áreas de Importancia Estratégica para la Conservación del Recurso Hídrico (AIECRH) que abastecen sus acueductos.

2. **Pago por Servicios Ambientales (PSA):** El **Decreto Ley 870 de 2017** y el **Decreto 1007 de 2018** crean el marco para compensar económicamente a propietarios u ocupantes rurales por conservar, restaurar o usar sosteniblemente ecosistemas que proveen servicios ambientales. La **Ley 2320 de 2023** modernizó el Artículo 111 incorporando Soluciones Basadas en la Naturaleza (SbN) y adaptación al cambio climático.

**Ruptura estructural 2016:** El Acuerdo de Paz con las FARC marcó un punto de quiebre histórico: los compromisos de inversión ambiental post-conflicto inyectaron recursos hacia zonas antes inaccesibles, acelerando la expansión del PSA en regiones como el Amazonas, el Pacífico y los Andes orientales.

**Meta Colombia 2030:** El país se comprometió a tener **1 millón de hectáreas** bajo esquemas de PSA, como parte de sus compromisos de carbono neutro y biodiversidad (NDC y GBF).

## Marco normativo PSA

| Norma | Contenido clave |
|---|---|
| **Ley 99/1993 Art. 111** | Obligación del 1% de ingresos corrientes para AIECRH |
| **Ley 2320/2023** | Actualiza Art. 111: SbN, restauración, REAA obligatorio |
| **Decreto Ley 870/2017** | Principios y lineamientos del PSA en Colombia |
| **Decreto 1007/2018** | Modalidades de PSA, plazos (hasta 5 años renovables), gastos asociados |

## Instituciones clave

| Institución | Rol en PSA y predios de conservación |
|---|---|
| **MADS** | Política nacional PSA, REAA, definición de AIECRH |
| **IDEAM** | Monitoreo de coberturas, índices hídricos, ENA |
| **IGAC** | Catastro multipropósito, LADM_COL, linderos prediales |
| **CARs** | Administración de recursos del 1%, aprobación de acuerdos PSA |
| **BancO2** | Plataforma de mercado voluntario de carbono y PSA privado |
| **DNP** | Evaluación costo-efectividad de los esquemas de conservación |

## Preguntas que responde este notebook

1. ¿Cuántas hectáreas están bajo PSA en el área de análisis y cuál ha sido la trayectoria histórica?
2. ¿El Acuerdo de Paz de 2016 generó una ruptura estructural estadísticamente detectable en la expansión del PSA? (prueba F de Chow)
3. ¿A qué ritmo es necesario crecer para alcanzar la meta nacional de 1M ha para 2030?
4. ¿Qué proyección arrojan los modelos para el período 2025–2030?
5. ¿El NDVI (proxy de salud del ecosistema) confirma que las hectáreas bajo PSA mantienen cobertura vegetal adecuada?

> **Ficha de dominio completa:** [`docs/fuentes/predios_conservacion.md`](../../docs/fuentes/predios_conservacion.md)  
> **Decisiones metodológicas:** [`docs/decisiones.md`](../../docs/decisiones.md)

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "predios_conservacion"
VARIABLE = "ndvi"
UNIDAD = ""

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `predios_conservacion` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "predios_conservacion.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos

### Fuentes de datos para hectáreas bajo PSA y NDVI en Colombia

La variable principal de esta línea son las **hectáreas bajo acuerdos PSA o predios adquiridos**. Las fuentes de datos son:

| Fuente | Variable | Frecuencia | Acceso |
|---|---|---|---|
| **REAA (MADS)** | Hectáreas registradas bajo AIECRH y PSA | Anual | SIRH / solicitud MADS |
| **RUNAP (Parques Nacionales)** | Áreas protegidas oficiales (ha) | Anual | [parquesnacionales.gov.co](https://www.parquesnacionales.gov.co) |
| **BancO2** | Hectáreas bajo PSA voluntario | Anual | [banco2.com.co](https://www.banco2.com.co) |
| **TerriData (DNP)** | Inversión del 1% por municipio | Anual | [terridata.dnp.gov.co](https://terridata.dnp.gov.co) |
| **Sentinel-2 / GEE** | NDVI promedio por polígono de predio | ~5 días | Google Earth Engine (libre) |

**Variable NDVI como proxy de efectividad:**
- El NDVI (Índice de Vegetación de Diferencia Normalizada) varía entre -1 y +1.
- Valores típicos para bosques: **0.5–0.8** (vegetación densa y saludable).
- Valores < 0.3 en predios bajo PSA son señal de alerta de pérdida de cobertura.
- Los satélites ópticos (Sentinel-2) tienen limitaciones por nubosidad en el Chocó y Amazonas; usar Sentinel-1 (radar) como complemento.

> La variable del notebook (`ndvi`) puede reemplazarse por `ha_psa` (hectáreas bajo PSA) según los datos disponibles. Colocar el archivo en `data/raw/` y ajustar la ruta.

In [ ]:
# df = load_csv("data/raw/predios_conservacion.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "ndvi": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA

### Rangos físicos y alertas de validación

La función `validate()` aplica rangos específicos para `predios_conservacion`:

- **NDVI:** Rango físico -1 a +1. Para predios bajo PSA, valores persistentemente < 0.2 son señal de pérdida de cobertura (posible incumplimiento del acuerdo).
- **Hectáreas PSA:** Siempre positivas. Un decremento brusco puede indicar vencimiento masivo de acuerdos o errores de registro.

**Riesgos de sesgo en los datos de PSA:**
- **Rezago catastral:** Predios pueden estar en PSA pero no actualizados en el REAA o LADM_COL.
- **Sesgo óptico:** En el Amazónico y Chocó, la nubosidad supera el 50% de las imágenes Sentinel-2 entre mayo y octubre. El NDVI de esos meses puede estar subestimado por nubes residuales.
- **Subregistro:** No todos los acuerdos PSA de CARs están consolidados en el REAA nacional.

> **ADR-002:** Los valores de NDVI muy bajos en predios de conservación no se eliminan. Son la señal más importante para detectar incumplimiento de los acuerdos PSA y deben desencadenar acciones de supervisión.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_predios_conservacion.html",
       title="EDA — Predios para Conservación", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

### Qué buscar en la visualización de hectáreas PSA y NDVI

La trayectoria histórica de las hectáreas bajo PSA es el indicador más directo de la efectividad de la política de conservación. Al visualizar:

- **Tendencia de largo plazo:** ¿Crece de forma sostenida el área bajo conservación? ¿La tasa de crecimiento es compatible con la meta de 1M ha para 2030?
- **Ruptura en 2016:** Buscar un quiebre visible en la pendiente de la curva alrededor de 2016 (efecto Acuerdo de Paz + políticas de inversión ambiental post-conflicto).
- **Línea de meta 2030:** Agregar una línea horizontal en 1.000.000 ha para visualizar la brecha entre el nivel actual y la meta nacional.
- **NDVI temporal:** Si se analiza el NDVI de predios específicos, la estacionalidad refleja los ciclos vegetativos. Un descenso sostenido en el NDVI medio de un conjunto de predios bajo PSA es evidencia de degradación del servicio ambiental.
- **Estacionalidad del NDVI:** Colombia tiene un patrón bimodal de productividad vegetal relacionado con las temporadas húmedas (MAM y SON). El NDVI sube en temporadas lluviosas y baja en períodos secos — esto es normal y no debe confundirse con degradación.

In [ ]:
plot_series(df, "fecha", "ndvi", title="Predios para Conservación — ndvi ()")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "ndvi", period="month")
plt.show()

## 3c. AHP para priorizacion de predios PSA — Costo de oportunidad

El **AHP (Analytic Hierarchy Process)** sopesa criterios ambientales y socioeconomicos para priorizar donde invertir recursos del 1% (Ley 99/1993 Art.111 y Ley 2320/2023):

| Criterio | Peso AHP | Por que |
|---|---|---|
| NDVI (vigor vegetal) | 35% | Mayor cobertura = menor intervencion requerida |
| TWI (indice topografico humedad) | 25% | Alta TWI = zona de recarga prioritaria |
| Costo de oportunidad ($/ha) | 20% | Menor costo = mayor eficiencia del PSA |
| Pendiente del terreno | 10% | Alta pendiente = riesgo erosion sin cobertura |
| Acceso a fuente de agua | 10% | Abastece acueducto municipal |

**Costo de oportunidad:** beneficio economico sacrificado al dejar la ganaderia/agricultura por conservacion. PSA eficiente: compensacion >= costo oportunidad.

**RBC (Relacion Beneficio-Costo):** RBC > 1 = el valor del servicio hidrico supera el costo del incentivo.

In [ ]:
# AHP para priorizacion de predios PSA
np.random.seed(17)
n_predios = 20

# Variables por predio (simuladas, normalizadas 0-1)
predios_df = pd.DataFrame({
    'predio_id': [f'PRD-{i+1:03d}' for i in range(n_predios)],
    'ndvi':           np.clip(np.random.normal(0.55, 0.18, n_predios), 0, 1),
    'twi':            np.clip(np.random.normal(0.5, 0.2, n_predios), 0, 1),
    'pendiente':      np.clip(np.random.normal(0.4, 0.2, n_predios), 0, 1),
    'acceso_agua':    np.random.uniform(0, 1, n_predios),
    'costo_oport_ha': np.random.uniform(300_000, 2_500_000, n_predios).round(-3),  # $/ha/ano
})

# Normalizar costo de oportunidad (inverso: menor costo = mayor puntaje)
co_min, co_max = predios_df['costo_oport_ha'].min(), predios_df['costo_oport_ha'].max()
predios_df['costo_norm'] = 1 - (predios_df['costo_oport_ha'] - co_min) / (co_max - co_min)

# Pesos AHP
pesos = {'ndvi': 0.35, 'twi': 0.25, 'costo_norm': 0.20, 'pendiente': 0.10, 'acceso_agua': 0.10}

# Puntaje AHP ponderado
predios_df['puntaje_ahp'] = sum(
    predios_df[col] * peso for col, peso in pesos.items())
predios_df['puntaje_ahp'] = predios_df['puntaje_ahp'].round(4)
predios_df['rango_ahp'] = predios_df['puntaje_ahp'].rank(ascending=False).astype(int)

# RBC simplificado: valor servicio hidrico / costo PSA
VALOR_AGUA_HA = 1_500_000  # $/ha/ano valor agua para acueducto
INCENTIVO_PSA = 600_000    # $/ha/ano incentivo tipico PSA (Decreto 1007/2018)
predios_df['rbc'] = (VALOR_AGUA_HA * predios_df['ndvi']) / predios_df['costo_oport_ha']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Puntaje AHP por predio
top10 = predios_df.nlargest(10, 'puntaje_ahp')
ax = axes[0]
bars = ax.barh(top10['predio_id'], top10['puntaje_ahp'],
               color=['#27ae60' if r <= 5 else '#f1c40f' for r in top10['rango_ahp']], alpha=0.85)
ax.set_xlabel('Puntaje AHP (0-1)')
ax.set_title('Top 10 Predios Priorizados — AHP\n(NDVI 35%, TWI 25%, Costo 20%...)', fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Panel B: Scatter NDVI vs Costo de oportunidad coloreado por RBC
ax = axes[1]
sc = ax.scatter(predios_df['costo_oport_ha']/1e6,
                predios_df['ndvi'],
                c=predios_df['rbc'], cmap='RdYlGn', s=60, alpha=0.8)
plt.colorbar(sc, ax=ax, label='RBC (valor/costo)')
ax.axhline(0.5, color='green', ls='--', lw=1)
ax.set_xlabel('Costo de oportunidad (M$/ha/ano)')
ax.set_ylabel('NDVI (vigor vegetal)')
ax.set_title('NDVI vs Costo oportunidad (color=RBC)\n(PSA eficiente: RBC > 1)', fontweight='bold')
ax.grid(alpha=0.3)

plt.suptitle('Priorizacion PSA — Ley 99/1993 Art.111 + Ley 2320/2023 + Decreto 1007/2018',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

n_rbc_positivo = (predios_df['rbc'] >= 1.0).sum()
print(f'Predios con RBC >= 1 (PSA eficiente): {n_rbc_positivo}/{n_predios}')
print(f'Costo oportunidad promedio: ${predios_df["costo_oport_ha"].mean()/1e6:.2f} M/ha/ano')
print('Top 3 predios priorizados:')
print(predios_df.nsmallest(3, 'rango_ahp')[['predio_id','puntaje_ahp','costo_oport_ha','rbc']])

## 4. Estadística descriptiva

### Indicadores de seguimiento para PSA y predios de conservación

| Estadístico | Interpretación en PSA |
|---|---|
| **Total ha acumuladas** | Punto de partida frente a la meta de 1M ha 2030 |
| **Tasa de crecimiento anual (ha/año)** | ¿Es suficiente para cerrar la brecha con la meta? |
| **NDVI medio de predios bajo PSA** | Salud promedio del ecosistema protegido |
| **Coeficiente de variación NDVI** | Alta variación entre predios indica heterogeneidad en la efectividad |
| **% hectáreas con NDVI < 0.3** | Proxy de proporción de predios con posible incumplimiento |

**Referencia de NDVI para Colombia (Sentinel-2, Landsat):**

| Cobertura | NDVI típico |
|---|---|
| Bosque denso (Amazónico, Andino) | 0.6–0.8 |
| Bosque secundario / rastrojo alto | 0.4–0.6 |
| Pastizal con cobertura natural | 0.2–0.4 |
| Suelo desnudo / quemado | 0.0–0.1 |
| Agua | < 0 |

**Umbral de alerta PSA:** NDVI < 0.3 en un predio bajo PSA durante más de 3 meses consecutivos activa revisión del acuerdo según los protocolos del MADS (Decreto 1007/2018).

In [ ]:
summarize(df)

## 5. Inferencial

### Estacionariedad y detección de ruptura estructural (F de Chow)

Para esta línea, el análisis inferencial central tiene dos componentes:

**1. Estacionariedad (ADR-004):**
- Las hectáreas bajo PSA son típicamente **no estacionarias** (tendencia creciente sostenida), lo cual es esperable y deseable para una política en expansión.
- La prueba ADF confirmará no estacionariedad (p > 0.05). Aplicar diferenciación (d=1) antes de modelar con ARIMA si fuera necesario.
- Para el NDVI, la estacionariedad depende de si hay tendencia de largo plazo en la salud del ecosistema. Una tendencia decreciente en el NDVI de predios bajo PSA es una señal de alerta crítica.

**2. Detección de ruptura estructural con F de Chow (2016):**

La **prueba F de Chow** evalúa si los parámetros de una regresión de tendencia difieren significativamente antes y después de un punto de corte conocido (2016 para esta línea):

- **H₀:** No hay ruptura estructural — la tasa de expansión del PSA es la misma antes y después de 2016.
- **H₁:** Existe ruptura — la pendiente y/o intercepto cambiaron en 2016.
- Si F es significativa (p < 0.05) → el Acuerdo de Paz alteró estadísticamente la trayectoria del PSA.

```python
# Implementación manual de F de Chow
import numpy as np
from scipy import stats

def chow_test(y, x, breakpoint_idx):
    """F de Chow para ruptura estructural en el índice breakpoint_idx."""
    y1, x1 = y[:breakpoint_idx], x[:breakpoint_idx]
    y2, x2 = y[breakpoint_idx:], x[breakpoint_idx:]
    
    def rss(yi, xi):
        b = np.polyfit(xi, yi, 1)
        return np.sum((yi - np.polyval(b, xi))**2)
    
    rss_full = rss(y, x)
    rss_sub = rss(y1, x1) + rss(y2, x2)
    k = 2  # parámetros (intercepto + pendiente)
    n = len(y)
    F = ((rss_full - rss_sub) / k) / (rss_sub / (n - 2*k))
    p = 1 - stats.f.cdf(F, k, n - 2*k)
    return {"F": F, "p": p}
```

**Mann-Kendall también por subperíodos:** Igual que en OT, aplicar Mann-Kendall a las dos ventanas (pre/post 2016) para cuantificar el cambio en la tasa de expansión del PSA.

In [ ]:
ts = df.set_index("fecha")["ndvi"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas

### Umbrales para predios de conservación y PSA

El PSA no tiene normas de cumplimiento con valores cuantitativos legales a nivel de serie temporal. Sin embargo, los siguientes umbrales son relevantes para el seguimiento:

| Umbral | Valor | Norma / Fuente | Uso |
|---|---|---|---|
| **Meta PSA Colombia** | 1.000.000 ha | NDC Colombia 2030 | Comparar avance acumulado |
| **Inversión mínima 1%** | 1% ingresos corrientes | Ley 99/1993 Art. 111 | Seguimiento financiero municipal |
| **NDVI mínimo bosque** | ≥ 0.5 | Protocolo MADS/IDEAM | Efectividad del PSA |
| **NDVI alerta pérdida** | < 0.3 (sostenido) | Decreto 1007/2018 | Activación de supervisión |

**Análisis de brechas (gap analysis):**
El `exceedance_report` se puede usar inversamente: en lugar de contar excedencias sobre un umbral máximo, calcular cuántos períodos el área acumulada de PSA **no alcanzó** una meta intermedia de crecimiento. Por ejemplo, si la meta requiere crecer 100.000 ha/año para llegar a 1M ha en 2030, ¿cuántos años se quedaron por debajo de este ritmo?

In [ ]:
rep = exceedance_report(df["ndvi"], variable="ndvi")
if rep.empty:
    print("Sin normas colombianas registradas para 'ndvi'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

### Tratamiento de datos para hectáreas PSA y NDVI

**Para hectáreas bajo PSA (serie anual):**
- Los faltantes son raros ya que el registro administrativo es el principal insumo.
- Si hay años sin reporte, verificar si realmente no se realizaron nuevos acuerdos o si hubo problemas de registro en el REAA.
- La interpolación lineal es apropiada para faltantes aislados.

**Para NDVI (serie mensual de imágenes satelitales):**
- La nubosidad tropical genera valores espurios (valores muy bajos que no corresponden a vegetación real sino a nubes).
- Estrategia recomendada: usar el **percentil 90** del NDVI en el período para cada polígono, en lugar de la media. El percentil 90 corresponde a las observaciones sin nubes y representa mejor la cobertura real.
- Alternativamente, fusionar datos Sentinel-2 (óptico) con Sentinel-1 (SAR, no afectado por nubes) usando técnicas de data fusion.

> **ADR-002:** El NDVI bajo por nubosidad NO debe eliminarse sin verificación. Distinguir entre: (a) pixeles enmascarados con calidad < 60% (filtrar) y (b) valores bajos reales por deforestación (conservar). El metadato de calidad de Sentinel-2 (banda SCL) permite esta distinción.

In [ ]:
df_clean = impute(df.copy(), cols=["ndvi"], method="linear")
print(f"Faltantes antes: {df["ndvi"].isna().sum()} | "
      f"después: {df_clean["ndvi"].isna().sum()}")

## 7. Modelos predictivos

### Proyección de hectáreas bajo PSA hacia la meta 2030

El objetivo predictivo central es proyectar las hectáreas bajo PSA al período **2025–2030** y comparar con la meta nacional de 1 millón de hectáreas.

**Opciones de modelado:**

| Modelo | Adecuado para | Limitación principal |
|---|---|---|
| **Random Forest** | Series con ruptura estructural (usar variable dummy 2016) | Requiere ≥20 observaciones para ser estable |
| **XGBoost** | Igual que RF; mejor rendimiento con muchas covariables | Sobreajuste con series cortas |
| **Regresión segmentada** | Proyección de tendencia con ruptura explícita en 2016 | Supone continuación lineal de la tendencia post-2016 |

**Covariables útiles para el modelo:**
- `dummy_post_2016`: Variable binaria (0 antes de 2016, 1 desde 2016).
- `inversion_millones_cop`: Inversión del 1% anual en el municipio/región.
- `precio_carbono_voluntario`: Precio de mercado en BancO2 (incentivo externo al PSA).

**Comparación con meta 2030:**
```python
META_2030 = 1_000_000  # hectáreas (NDC Colombia 2030)
anio_actual = 2024
ha_actual = df["ha_psa"].iloc[-1]
ha_faltante = META_2030 - ha_actual
anios_restantes = 2030 - anio_actual
ritmo_requerido = ha_faltante / anios_restantes
print(f"Ritmo requerido: {ritmo_requerido:,.0f} ha/año para alcanzar la meta de 2030")
```

**Métricas de evaluación (ADR-003):** RMSE y MAE. No usar RMSLE para hectáreas (la variable puede tener valores muy pequeños en regiones con PSA naciente, sesgando el error logarítmico).

In [ ]:
ts = df_clean.set_index("fecha")["ndvi"]

models = {
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Conclusiones

### Plantilla de conclusiones (completar con datos reales)

**Situación actual:**
- Hectáreas bajo PSA al último año disponible: `X` ha (de la meta 2030: `Y%` alcanzado).
- NDVI medio de predios bajo PSA: `X` (saludable ≥ 0.5; en degradación si < 0.3).

**Ruptura estructural 2016 (F de Chow):**
- F = `X`, p = `Y`. `[Se confirma / No se confirma]` ruptura estructural en 2016.
- Tasa de expansión pre-2016: `X` ha/año. Post-2016: `Y` ha/año.
- Conclusión: el Acuerdo de Paz `[aceleró / no cambió significativamente]` la expansión del PSA.

**Tendencia (Mann-Kendall):**
- Serie completa: `[creciente / decreciente / sin tendencia]`, slope = `X` ha/año (p = `Y`).

**Proyección 2025–2030:**
- Modelo seleccionado: `[nombre]` con RMSE = `X` ha.
- Proyección 2030: `X` ha. ¿Compatible con la meta de 1M ha? `[Sí / No — requiere acelerar el ritmo a X ha/año]`.

### Normativa y referencias

- **Ley 99/1993 Art. 111:** Obligación del 1% para AIECRH.
- **Decreto 1007/2018:** Marco del PSA — modalidades y plazos.
- **Ley 2320/2023:** Actualización Art. 111 con SbN y REAA.
- **NDC Colombia 2030:** Meta de 1M ha bajo PSA.
- Registrar decisiones metodológicas en [`docs/decisiones.md`](../../docs/decisiones.md).

## 9. Cómo adaptar a datos reales

### Paso 1: Obtener datos de hectáreas bajo PSA

```python
# Opción A: REAA / MADS (base oficial de PSA en Colombia)
# Solicitar al MADS o consultar el SIRH por acceso institucional
df = load_csv("data/raw/psa_hectareas_region.csv", date_col="año")
# Columnas esperadas: año, ha_psa (hectáreas nuevas o acumuladas), municipio

# Opción B: TerriData (DNP)
# Indicador: "Inversión recursos del 1% ingresos corrientes - Art. 111 Ley 99/1993"
# URL: https://terridata.dnp.gov.co/

# Opción C: BancO2 (mercado voluntario)
# Hectáreas vinculadas al mercado voluntario de carbono
# URL: https://www.banco2.com.co
```

### Paso 2: Ajustar la variable principal

Si se trabaja con hectáreas bajo PSA (no NDVI), cambiar en Setup:
```python
VARIABLE = "ha_psa"    # hectáreas bajo PSA acumuladas
UNIDAD = "ha"
```

### Paso 3: Detección de ruptura estructural con F de Chow

```python
import numpy as np
from scipy import stats

# Preparar series
ts = df.set_index("fecha")["ha_psa"].dropna()
x = np.arange(len(ts))
y = ts.values

# Índice del punto de ruptura (2016)
breakpoint_idx = ts.index.get_loc("2016")  # Ajustar según datos

# F de Chow
def chow_test(y, x, bp):
    def rss_fit(yi, xi): return np.sum((yi - np.polyval(np.polyfit(xi, yi, 1), xi))**2)
    rss_full = rss_fit(y, x)
    rss_sub = rss_fit(y[:bp], x[:bp]) + rss_fit(y[bp:], x[bp:])
    k, n = 2, len(y)
    F = ((rss_full - rss_sub) / k) / (rss_sub / (n - 2*k))
    return {"F": F, "p": 1 - stats.f.cdf(F, k, n - 2*k)}

resultado = chow_test(y, x, breakpoint_idx)
print(f"F de Chow: F={resultado['F']:.3f}, p={resultado['p']:.4f}")
```

### Paso 4: Comparación con meta nacional 2030

```python
META_2030 = 1_000_000  # ha (NDC Colombia 2030)

ha_actual = df["ha_psa"].iloc[-1]
anio_actual = df["fecha"].iloc[-1].year
anios_restantes = 2030 - anio_actual
ritmo_requerido = (META_2030 - ha_actual) / anios_restantes

print(f"Ha actuales: {ha_actual:,.0f}")
print(f"Brecha 2030: {META_2030 - ha_actual:,.0f} ha")
print(f"Ritmo requerido: {ritmo_requerido:,.0f} ha/año")
print(f"Progreso: {ha_actual/META_2030*100:.1f}% de la meta")
```

### Paso 5: Validar efectividad con NDVI de predios

```python
# Si se tienen datos de NDVI de los predios bajo PSA:
df_ndvi = load_csv("data/raw/ndvi_predios_psa.csv", date_col="fecha")

# Alerta: NDVI < 0.3 indica posible pérdida de cobertura
alerta = df_ndvi[df_ndvi["ndvi"] < 0.3]
print(f"Predios con alerta de pérdida de cobertura: {len(alerta)} registros")
```

### Paso 6: Registrar decisiones

Al finalizar el análisis, documentar en [`docs/decisiones.md`](../../docs/decisiones.md):
- Si se detectó ruptura estructural en 2016 y su magnitud.
- El modelo seleccionado y la proyección 2030.
- Si el ritmo actual es compatible con la meta nacional.
- Alertas de NDVI bajo en predios específicos (para coordinación con CARs).